In [95]:
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup, Tag
import pandas as pd

In [96]:
# HTML lokal
HTML_CONTOH = """<!doctype html><html lang="id">  <head><title>Toko Buku Data</title></head>  <body>    <h1>Buku Pilihan</h1>    <section id="katalog">      <article class="buku unggulan" data-id="B001">        <h2 class="judul">Dasar Data Mining</h2>        <p class="harga">Rp125.000</p>        <p class="stok">Tersedia</p>        <a href="/buku/dasar-data-mining">Detail</a>      </article>      <article class="buku" data-id="B002">        <h2 class="judul">Python untuk Analisis Data</h2>        <p class="harga">Rp149.500</p>        <p class="stok habis">Habis</p>        <a href="/buku/python-analisis">Detail</a>      </article>      <article class="buku" data-id="B003">        <h2 class="judul">Statistika Praktis</h2>        <p class="harga">Rp98.000</p>        <!-- Elemen stok sengaja tidak tersedia -->        <a href="/buku/statistika-praktis">Detail</a>      </article>    </section>  </body></html>"""
soup = BeautifulSoup(HTML_CONTOH, "html.parser")

In [97]:
BASE_URL_CONTOH = "https://contoh.invalid"
def teks_atau_none(induk: Tag, selector: str):
    elemen = induk.select_one(selector)
    return elemen.get_text(" ", strip=True) if elemen else None

def harga_ke_int(teks_harga: str):
    if teks_harga is None:
        return None
    digit = "".join(karakter for karakter in teks_harga if karakter.isdigit())
    return int(digit) if digit else None

In [98]:
# Nomor 1
buku_tersedia = [
    teks_atau_none(kartu, ".judul")
    for kartu in soup.select("article.buku")
    if teks_atau_none(kartu, ".stok") == "Tersedia"
]
print("1. Judul buku tersedia:", buku_tersedia)

1. Judul buku tersedia: ['Dasar Data Mining']


In [99]:
# Nomor 2
daftar_harga = [
    harga_ke_int(teks_atau_none(kartu, ".harga"))
    for kartu in soup.select("article.buku")
]
harga_valid = [h for h in daftar_harga if h is not None]
rata_rata_harga = sum(harga_valid) / len(harga_valid) if harga_valid else 0
print(f"2. Rata-rata harga buku: Rp{rata_rata_harga:,.2f}")

2. Rata-rata harga buku: Rp124,166.67


In [100]:
# Nomor 3
semua_data_id = [tag.get("data-id") for tag in soup.select("article[data-id]")]
print("3. Daftar data-id:", semua_data_id)

3. Daftar data-id: ['B001', 'B002', 'B003']


In [101]:
# Nomor 4
def ekstrak_buku_latihan(dokumen: BeautifulSoup):
    def ekstrak_satu(kartu: Tag):
        tautan = kartu.select_one("a[href]")
        href = tautan.get("href") if tautan else None
        # "Tidak diketahui"
        stok_teks = teks_atau_none(kartu, ".stok")
        if stok_teks is None:
            stok_teks = "Tidak diketahui"
        return {
            "id": kartu.get("data-id"),
            "judul": teks_atau_none(kartu, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(kartu, ".harga")),
            "stok": stok_teks,
            "url": urljoin(BASE_URL_CONTOH, str(href)) if href else None,
        }
    return tuple(ekstrak_satu(kartu) for kartu in dokumen.select("article.buku"))

In [102]:
print("1. Buku yang tersedia:", ", ".join(buku_tersedia))
h_format = f"{rata_rata_harga:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
print(f"2. Rata-rata harga buku: Rp{h_format}")
print("3. Daftar atribut data id:", ", ".join(semua_data_id))
print("4. Hasil DataFrame setelah fungsi diubah:")
df_latihan = pd.DataFrame(ekstrak_buku_latihan(soup))
df_display = df_latihan.rename(columns={
    "id": "ID",
    "judul": "Judul Buku",
    "harga_rupiah": "Harga (Rp)",
    "stok": "Status Stok",
    "url": "Link URL"
})
df_display.style.format(
    {"Harga (Rp)": "Rp{:,.0f}"}
).hide(
    axis="index"
).set_properties(
    **{
        "text-align": "left",
        "padding": "8px 12px",
        "white-space": "nowrap"
    }
).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
).set_caption(
    "<!-- processed by catherine -->" 
)

1. Buku yang tersedia: Dasar Data Mining
2. Rata-rata harga buku: Rp124.166,67
3. Daftar atribut data id: B001, B002, B003
4. Hasil DataFrame setelah fungsi diubah:


ID,Judul Buku,Harga (Rp),Status Stok,Link URL
B001,Dasar Data Mining,"Rp125,000",Tersedia,https://contoh.invalid/buku/dasar-data-mining
B002,Python untuk Analisis Data,"Rp149,500",Habis,https://contoh.invalid/buku/python-analisis
B003,Statistika Praktis,"Rp98,000",Tidak diketahui,https://contoh.invalid/buku/statistika-praktis
